# Calibrated PPI example

One calibrated PPI run: data and split -> target -> inference -> output.

Run from `examples/simulation/`, so that `dataset` and `model` import directly.
Calibeating Prediction-Powered Inference (van der Laan & van der Laan, 2026), Algorithms 1–2. The original model is independently trained; the calibrator intentionally uses the labeled inference sample. The estimate pools both sets of calibrated predictions, and the default SE uses the paper’s squared influence values.


In [1]:
from sklearn.ensemble import RandomForestRegressor

from openppi import Mean, PPIData, get_method, split
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Split iid rows; fit only on the independent training sample.
frame = make_judge_data(n=2000, seed=0)
train, labeled, unlabeled = split(
    frame.drop(columns='label').to_numpy(), frame['label'].to_numpy(),
    train_size=400, labeled_size=400, random_state=42)
model = RandomForestRegressor(n_estimators=60, min_samples_leaf=8,
                              max_depth=8, random_state=0, n_jobs=1)
model.fit(*train)
data = PPIData(labels=labeled[1], labeled_predictions=model.predict(labeled[0]),
               unlabeled_predictions=model.predict(unlabeled))

In [3]:
# 2. Select the target.
target = Mean()

In [4]:
# 3. Fit isotonic calibration on labeled scores, then pool calibrated predictions.
result = get_method('CalPPI').inference(data, target=target, alpha=.05)

In [5]:
# 4. Output.
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
print('Isotonic blocks:', result.diagnostics['calibration_parameters']['block_count'])
print('Mean calibration residual:', result.diagnostics['calibration_residual_mean'])

Estimate: 0.49449526670904376
Confidence interval: (0.44859369938116267, 0.5403968340369248)
Isotonic blocks: 15
Mean calibration residual: -1.3322676295501878e-17


In [6]:
# Compare with affine calibration (Algorithm 2).
linear = get_method('LinearCalPPI').inference(data, target=target, alpha=.05)
print('Linear estimate:', linear.estimate)
print('Linear confidence interval:', linear.confidence_interval)
print('Affine fit:', linear.diagnostics['calibration_parameters'])


Linear estimate: 0.4965965791569647
Linear confidence interval: (0.44998275593763465, 0.5432104023762947)
Affine fit: {'intercept': 0.11003495837831628, 'slope': 0.7447844217448726}
